<a href="https://colab.research.google.com/github/dtoralg/TheValley_MDS/blob/main/%5B02%5D%20-%20Analisis_Cluster/%5B01%5D%20-%20Notebooks/E5_Comparacion_de_modelos_de_clustering_MALL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ejercicio: Segmentación de clientes basada en comportamiento de compra (Mall Customers Dataset)

 **Objetivo**:
Aplicar y comparar distintos algoritmos de clustering (K-Means, Hierarchical, DBSCAN) para segmentar clientes en función de su perfil de gasto y características demográficas.


### Dataset:

Mall Customers (kaggle)

Observaciones: 200 clientes

Variables:

* CustomerID: ID del cliente

* Gender: Género (categoría)

* Age: Edad

* Annual Income (k$): Ingreso anual

* Spending Score (1-100): Puntuación de gasto (basado en comportamiento de compra)


## Aplicación de algoritmos

### K-Means
* Elegir número óptimo de clústeres (Elbow, Silhouette)

* Asignar etiquetas

* Visualización 2D con color por clúster

### Clustering jerárquico
* Crear dendrograma (scipy)

* Probar con linkage: ward, complete, average

* Cortar el dendrograma en el mismo número de clústeres que K-Means

* Comparar resultados

### DBSCAN
* Buscar parámetros óptimos eps y min_samples

* Identificar clústeres + outliers

* Visualizar y comparar (ver si capta mejor la forma no esférica de los grupos)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import kagglehub

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import dendrogram, linkage

import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Download latest version
path = kagglehub.dataset_download("vjchoudhary7/customer-segmentation-tutorial-in-python")

print("Path to dataset files:", path)

df = pd.read_csv("/kaggle/input/customer-segmentation-tutorial-in-python/Mall_Customers.csv")

In [ ]:
print(df.info())
print(df.describe())

# Eliminamos CustomerID (no aporta al clustering)
df.drop(columns=['CustomerID'], inplace=True)

# Codificamos 'Gender'
df['Gender'] = df['Gender'].map({'Male': 0, 'Female': 1})

In [ ]:
sns.scatterplot(data=df, x='Annual Income (k$)', y='Spending Score (1-100)', hue='Gender')
plt.title('Distribución de ingreso vs gasto')
plt.show()


In [ ]:
X = df.copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
inertia = []
silhouette = []
K = range(2, 11)

for k in K:
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(X_scaled)
    inertia.append(kmeans.inertia_)
    silhouette.append(silhouette_score(X_scaled, kmeans.labels_))

plt.plot(K, inertia, marker='o')
plt.title('Método del codo')
plt.xlabel('Número de clusters'); plt.ylabel('Inercia')
plt.show()

plt.plot(K, silhouette, marker='o', color='green')
plt.title('Silhouette score')
plt.xlabel('Número de clusters'); plt.ylabel('Silhouette')
plt.show()

In [ ]:
kmeans = KMeans(n_clusters=5, random_state=42)
df['KMeans_Label'] = kmeans.fit_predict(X_scaled)

sns.scatterplot(data=df, x='Annual Income (k$)', y='Spending Score (1-100)', hue='KMeans_Label', palette='tab10')
plt.title('K-Means Clustering')
plt.show()

In [ ]:
linked = linkage(X_scaled, method='ward')

plt.figure(figsize=(10, 5))
dendrogram(linked, truncate_mode='lastp', p=20)
plt.title('Dendrograma - clustering jerárquico')
plt.xlabel('Clientes'); plt.ylabel('Distancia')
plt.show()

In [ ]:
hierarchical = AgglomerativeClustering(n_clusters=5)
df['Hierarchical_Label'] = hierarchical.fit_predict(X_scaled)

sns.scatterplot(data=df, x='Annual Income (k$)', y='Spending Score (1-100)', hue='Hierarchical_Label', palette='Set2')
plt.title('Clustering Jerárquico')
plt.show()

In [ ]:
# Parámetros típicos: eps=0.5 y min_samples=5, pero se pueden ajustar
db = DBSCAN(eps=0.8, min_samples=5)
df['DBSCAN_Label'] = db.fit_predict(X_scaled)

sns.scatterplot(data=df, x='Annual Income (k$)', y='Spending Score (1-100)', hue='DBSCAN_Label', palette='Set1')
plt.title('DBSCAN Clustering')
plt.show()

In [ ]:
print("KMeans Silhouette:", silhouette_score(X_scaled, df['KMeans_Label']))
print("Hierarchical Silhouette:", silhouette_score(X_scaled, df['Hierarchical_Label']))
# DBSCAN podría tener -1 (outliers)
labels_db = df['DBSCAN_Label']
mask = labels_db != -1  # Solo clústeres válidos

if len(set(labels_db)) > 1 and sum(mask) > 1:
    print("DBSCAN Silhouette:", silhouette_score(X_scaled[mask], labels_db[mask]))
else:
    print("DBSCAN no encontró suficientes clústeres para calcular Silhouette")

In [ ]:
nuevo_cliente = np.array([[0, 40, 60, 50]])  # Ej: hombre, 40 años, 60k ingreso, 50 de score
nuevo_cliente_scaled = scaler.transform(nuevo_cliente)

cluster = kmeans.predict(nuevo_cliente_scaled)[0]
print(f"Nuevo cliente clasificado en el clúster {cluster} por K-Means")